# 02 — Lp(a) correlation pairs (NHANES III Phase II)

Estimates the 6 correlations of Lp(a) against the rest of the matrix:

- Lp(a) ↔ {BMI, LDL-C, SBP, FPG (fasting), smoking, eGFR}

**Lp(a) is only measured in NHANES III Phase II (1991–94)** — the variable `LPP` lives in `lab.dat` (Release 1A) and was populated only for Phase II respondents. So this notebook parses three NHANES III source files (`lab.dat`, `exam.dat`, `adult.dat`), merges by `SEQN`, and computes weighted Spearman correlations against the trial-band 65–80 slice with 95 % CIs from a paired-PSU jackknife.

Time-trend stratification is **not** possible (single 4-year cycle); age stratification is reported as a stability check.

In [1]:
import os, warnings
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')

DATA = Path(os.path.abspath(os.path.join('..', 'data')))
RAW3 = DATA / 'raw' / 'nhanes' / 'nhanes3'
OUT = Path('outputs'); OUT.mkdir(parents=True, exist_ok=True)

## 1. Parse the three NHANES III source files

Column positions come from the official SAS read-in scripts at
`https://wwwn.cdc.gov/nchs/data/nhanes3/1a/{lab,exam,adult}.sas`
(positions 1-based, inclusive in SAS; pandas `read_fwf` takes 0-based
half-open slices, hence the `(start-1, stop)` translation).

Variables kept:

- **`lab.dat`** (Release 1A, ~58 MB):
  `SEQN` (1–5), `HSSEX` (15), `HSAGEIR` (16–17), `SDPPHASE` (40),
  `SDPPSU6` (41), `SDPSTRA6` (42–43), `WTPFEX6` (59–67),
  `PHPFAST` (1263–1267), `LCP` (1615–1617), `CEP` (1784–1787),
  `G2P` (1885–1889), `LPP` (1643–1645).
- **`exam.dat`** (~195 MB): `SEQN` (1–5), `PEPMNK1R` (1423–1425),
  `BMPBMI` (1524–1527).
- **`adult.dat`** (~67 MB): `SEQN` (1–5), `HAR1` (2281), `HAR3` (2285).

In [2]:
LAB_COLS = [
    ((1, 5),       'SEQN'),
    ((15, 15),     'HSSEX'),
    ((16, 17),     'HSAGEIR'),
    ((18, 18),     'HSAGEU'),
    ((40, 40),     'SDPPHASE'),
    ((41, 41),     'SDPPSU6'),
    ((42, 43),     'SDPSTRA6'),
    ((59, 67),     'WTPFEX6'),
    ((1263, 1267), 'PHPFAST'),
    ((1615, 1617), 'LCP'),       # LDL-C mg/dL
    ((1784, 1787), 'CEP'),       # Serum creatinine mg/dL
    ((1885, 1889), 'G2P'),       # Plasma glucose mg/dL
    ((1643, 1645), 'LPP'),       # Lp(a) mg/dL
]
specs = [(s - 1, e) for (s, e), _ in LAB_COLS]
names = [n for _, n in LAB_COLS]
lab = pd.read_fwf(RAW3 / 'lab.dat', colspecs=specs, names=names, dtype=str)

# All numeric variables here are stored either as digits (e.g. "00142")
# or with an explicit decimal ("142.0"); pd.to_numeric handles both.
# The lab.sas "8.2" / "6.1" formats are SAS *display* formats —
# the raw file has decimals embedded where present.
for col in ['HSAGEIR', 'SDPPSU6', 'SDPSTRA6', 'WTPFEX6', 'PHPFAST',
            'LCP', 'CEP', 'G2P', 'LPP']:
    lab[col] = pd.to_numeric(lab[col], errors='coerce')

# WTPFEX6 has 2 implicit decimals (no period in the file), so divide:
lab['WTPFEX6'] = lab['WTPFEX6'] / 100.0

# Missing-code handling per the NHANES III lab codebook
lab.loc[lab['LPP'] == 888, 'LPP'] = np.nan
lab.loc[lab['LCP'] == 888, 'LCP'] = np.nan
lab.loc[lab['CEP'] == 8888, 'CEP'] = np.nan
lab.loc[lab['G2P'] == 88888, 'G2P'] = np.nan
lab.loc[lab['PHPFAST'] == 88888, 'PHPFAST'] = np.nan

lab['HSSEX'] = lab['HSSEX'].map({'1': 'Male', '2': 'Female'})
lab['HSAGEU'] = lab['HSAGEU'].map({'1': 'Months', '2': 'Years'})
lab['SDPPHASE'] = lab['SDPPHASE'].map({'1': 'Phase 1 (1988-91)', '2': 'Phase 2 (1991-94)'})
lab['age_years'] = np.where(lab['HSAGEU'] == 'Years', lab['HSAGEIR'], np.nan)
lab['fasting'] = lab['PHPFAST'] >= 8.0
print(f'lab.dat parsed: {len(lab):,} rows; Phase II + adult + LPP-measured: '
      f'{((lab["SDPPHASE"]=="Phase 2 (1991-94)") & (lab["age_years"]>=20) & lab["LPP"].notna()).sum():,}')
print(f'  fasting (≥8h) Phase II adults: {((lab["SDPPHASE"]=="Phase 2 (1991-94)") & (lab["age_years"]>=20) & lab["fasting"]).sum():,}')

lab.dat parsed: 29,314 rows; Phase II + adult + LPP-measured: 8,217
  fasting (≥8h) Phase II adults: 4,979


In [3]:
EXAM_COLS = [
    ((1, 5),       'SEQN'),
    ((1423, 1425), 'PEPMNK1R'),    # mean SBP, mmHg (integer)
    ((1524, 1527), 'BMPBMI'),      # BMI, kg/m^2 (mixed decimal/integer)
]
specs = [(s - 1, e) for (s, e), _ in EXAM_COLS]
names = [n for _, n in EXAM_COLS]
exam = pd.read_fwf(RAW3 / 'exam.dat', colspecs=specs, names=names, dtype=str)
for col in ['PEPMNK1R', 'BMPBMI']:
    exam[col] = pd.to_numeric(exam[col], errors='coerce')
# 888 / 8888 = could not be obtained
exam.loc[exam['PEPMNK1R'] == 888, 'PEPMNK1R'] = np.nan
exam.loc[exam['BMPBMI'] == 8888, 'BMPBMI'] = np.nan
print(f'exam.dat parsed: {len(exam):,} rows')
print(f'  BMI present: {exam["BMPBMI"].notna().sum():,};  SBP present: {exam["PEPMNK1R"].notna().sum():,}')
print(f'  BMI mean (sanity): {exam["BMPBMI"].mean():.2f};  SBP mean: {exam["PEPMNK1R"].mean():.2f}')

exam.dat parsed: 31,311 rows
  BMI present: 27,830;  SBP present: 23,756
  BMI mean (sanity): 23.80;  SBP mean: 118.58


In [4]:
ADULT_COLS = [
    ((1, 5),     'SEQN', 'str'),
    ((2281, 2281), 'HAR1', 'int'),    # smoked 100+ cigarettes in life (1=Y, 2=N)
    ((2285, 2285), 'HAR3', 'int'),    # smoke now (1=Y, 2=N)
]
specs = [(s - 1, e) for (s, e), _, _ in ADULT_COLS]
names = [n for _, n, _ in ADULT_COLS]
adult = pd.read_fwf(RAW3 / 'adult.dat', colspecs=specs, names=names, dtype=str)
for (_, _), n, t in ADULT_COLS:
    if t == 'int':
        adult[n] = pd.to_numeric(adult[n], errors='coerce')
# Code smoking ordinal: 1 = current, 2 = former, 3 = never
def smoke_cat3(row):
    if row['HAR1'] == 2:
        return 3  # never
    if row['HAR1'] == 1:
        if row['HAR3'] == 1:
            return 1  # current
        if row['HAR3'] == 2:
            return 2  # former
    return np.nan
adult['smoking_cat'] = adult.apply(smoke_cat3, axis=1)
print(f'adult.dat parsed: {len(adult):,} rows; smoking_cat non-null: {adult["smoking_cat"].notna().sum():,}')

adult.dat parsed: 20,050 rows; smoking_cat non-null: 20,032


In [5]:
df = lab.merge(exam, on='SEQN', how='left').merge(adult, on='SEQN', how='left')

# CKD-EPI 2021 (race-free) using NHANES III creatinine
def ckd_epi_2021(scr, age, female):
    if pd.isna(scr) or pd.isna(age) or pd.isna(female):
        return np.nan
    kappa = 0.7 if female else 0.9
    alpha = -0.241 if female else -0.302
    sex_factor = 1.012 if female else 1.0
    ratio = scr / kappa
    return 142 * (min(ratio, 1) ** alpha) * (max(ratio, 1) ** -1.200) * (0.9938 ** age) * sex_factor

df['eGFR'] = df.apply(
    lambda r: ckd_epi_2021(r['CEP'], r['age_years'], r['HSSEX'] == 'Female'),
    axis=1,
)

# FPG: only valid if fasting (≥ 8 h)
df['FPG'] = np.where(df['fasting'], df['G2P'], np.nan)

# Restrict to Phase II adults with valid weight
ana = df[(df['SDPPHASE'] == 'Phase 2 (1991-94)')
          & (df['age_years'] >= 20)
          & (df['WTPFEX6'].fillna(0) > 0)].copy()

trial = ana[(ana['age_years'] >= 65) & (ana['age_years'] <= 80)].copy()
print(f'Phase II adults 20+:                                   n = {len(ana):,}')
print(f'  with all 6 risks non-null (BMI/LDL/SBP/FPG/smoke/eGFR): {ana[["LPP", "BMPBMI", "LCP", "PEPMNK1R", "FPG", "smoking_cat", "eGFR"]].notna().all(axis=1).sum():,}')
print(f'Trial-band 65-80, with Lp(a) measured:                 n = {trial["LPP"].notna().sum():,}')

Phase II adults 20+:                                   n = 8,360
  with all 6 risks non-null (BMI/LDL/SBP/FPG/smoke/eGFR): 1,509
Trial-band 65-80, with Lp(a) measured:                 n = 1,458


## 2. Weighted Spearman + paired-PSU jackknife (NHANES III variance design)

Same code as notebook 01, but the variance design uses NHANES III's `SDPSTRA6` × `SDPPSU6` instead of continuous NHANES's `SDMVSTRA` × `SDMVPSU`.

In [6]:
def weighted_rank(x, w):
    o = np.argsort(x, kind='stable')
    cw = np.cumsum(w[o])
    r = np.empty_like(x, dtype=float)
    r[o] = cw - w[o] / 2.0
    return r

def w_spearman(x, y, w):
    rx, ry = weighted_rank(x, w), weighted_rank(y, w)
    mx, my = np.average(rx, weights=w), np.average(ry, weights=w)
    cov = np.average((rx - mx) * (ry - my), weights=w)
    sx = np.sqrt(np.average((rx - mx) ** 2, weights=w))
    sy = np.sqrt(np.average((ry - my) ** 2, weights=w))
    if sx == 0 or sy == 0:
        return np.nan
    return float(cov / (sx * sy))

def paired_jackknife_spearman(df, x_col, y_col,
                              wt='WTPFEX6', psu='SDPPSU6', stratum='SDPSTRA6'):
    sub = df[[x_col, y_col, wt, psu, stratum]].dropna()
    sub = sub[sub[wt] > 0]
    if len(sub) < 30:
        return np.nan, np.nan, len(sub)
    x = sub[x_col].values.astype(float)
    y = sub[y_col].values.astype(float)
    w = sub[wt].values.astype(float)
    s = sub[stratum].astype(int).values
    p = sub[psu].astype(int).values
    theta = w_spearman(x, y, w)
    var_sum = 0.0
    for st in np.unique(s):
        in_str = s == st
        psus_in = np.unique(p[in_str])
        if len(psus_in) < 2:
            continue
        for psu_id in psus_in:
            wr = w.copy()
            mask_in = in_str & (p == psu_id)
            mask_other = in_str & (p != psu_id)
            wr[mask_in] = 0.0
            wr[mask_other] *= 2.0
            if wr.sum() <= 0:
                continue
            t = w_spearman(x, y, wr)
            if not np.isnan(t):
                var_sum += (t - theta) ** 2
    return theta, np.sqrt(var_sum), len(sub)

## 3. Sign convention + headline 6 pairs (trial band 65–80)

Same convention as notebook 01:

- BMI / LDL-C / SBP / FPG: bigger = worse → no flip.
- smoking_cat: 1 = current (worst), 3 = never (best). Flip to `4 - smoking_cat` so big = worse.
- eGFR: high = good. Flip to `-eGFR` so big = worse kidney function.
- Lp(a) (`LPP`): higher = worse cardiovascular risk. No flip.

In [7]:
trial['smoking_signed'] = 4 - trial['smoking_cat']
trial['eGFR_signed'] = -trial['eGFR']
ana['smoking_signed'] = 4 - ana['smoking_cat']
ana['eGFR_signed'] = -ana['eGFR']

OTHER_RISKS = {
    'BMI':                'BMPBMI',
    'LDL_C':              'LCP',
    'SBP':                'PEPMNK1R',
    'FPG':                'FPG',
    'smoking':            'smoking_signed',
    'kidney_dysfunction': 'eGFR_signed',
}
rows = []
for label, col in OTHER_RISKS.items():
    rho, se, n = paired_jackknife_spearman(trial, 'LPP', col)
    rows.append({
        'pair': f'lipoprotein_a ↔ {label}',
        'risk_a': 'lipoprotein_a',
        'risk_b': label,
        'n': n,
        'rho': rho,
        'se':  se,
        'lo':  rho - 1.96 * se if not pd.isna(se) else np.nan,
        'hi':  rho + 1.96 * se if not pd.isna(se) else np.nan,
    })
lpa_headline = pd.DataFrame(rows).round(3)
print('Trial-band 65-80, weighted Spearman, 95 % CI from paired-PSU jackknife:')
print(lpa_headline[['pair', 'n', 'rho', 'se', 'lo', 'hi']].to_string(index=False))

Trial-band 65-80, weighted Spearman, 95 % CI from paired-PSU jackknife:
                              pair    n    rho    se     lo     hi
               lipoprotein_a ↔ BMI 1454 -0.071 0.012 -0.095 -0.047
             lipoprotein_a ↔ LDL_C  601  0.145 0.019  0.107  0.182
               lipoprotein_a ↔ SBP 1455  0.058 0.013  0.033  0.084
               lipoprotein_a ↔ FPG  521 -0.122 0.016 -0.154 -0.090
           lipoprotein_a ↔ smoking 1458 -0.039 0.008 -0.054 -0.023
lipoprotein_a ↔ kidney_dysfunction 1444  0.032 0.013  0.005  0.058


## 4. Age stratification — point estimates only

(No time-trend stratification: NHANES III Phase II is a single 4-year cycle.)

In [8]:
AGE_BANDS = {'40-64': (40, 65), '65-80': (65, 81), '80+': (80, 200)}
rows = []
for label, col in OTHER_RISKS.items():
    row = {'pair': f'lipoprotein_a ↔ {label}'}
    for ag, (lo, hi) in AGE_BANDS.items():
        sub = ana[ana['age_years'].between(lo, hi - 1)]
        rho, _, n = paired_jackknife_spearman(sub, 'LPP', col)
        row[f'n_{ag}'] = n
        row[ag] = rho
    rows.append(row)
lpa_age = pd.DataFrame(rows)
lpa_age['range'] = (lpa_age[list(AGE_BANDS.keys())].max(axis=1)
                     - lpa_age[list(AGE_BANDS.keys())].min(axis=1))
cols = ['pair'] + sum([['n_'+k, k] for k in AGE_BANDS.keys()], []) + ['range']
print('Lp(a) pair correlations by age band:')
print(lpa_age[cols].round(3).to_string(index=False))
diverging = lpa_age[lpa_age['range'] > 0.10]
print(f'\n{len(diverging)} of {len(lpa_age)} pairs diverge by > 0.10 across age bands.')

Lp(a) pair correlations by age band:
                              pair  n_40-64  40-64  n_65-80  65-80  n_80+    80+  range
               lipoprotein_a ↔ BMI     2723 -0.064     1454 -0.071    560 -0.075  0.011
             lipoprotein_a ↔ LDL_C     1175  0.147      601  0.145    224  0.163  0.018
               lipoprotein_a ↔ SBP     2722 -0.006     1455  0.058    560  0.020  0.065
               lipoprotein_a ↔ FPG     1519 -0.032      521 -0.122      0    NaN  0.090
           lipoprotein_a ↔ smoking     2726 -0.020     1458 -0.039    561 -0.036  0.018
lipoprotein_a ↔ kidney_dysfunction     2703  0.016     1444  0.032    558  0.091  0.075

0 of 6 pairs diverge by > 0.10 across age bands.


In [9]:
lpa_headline.to_parquet(OUT / 'lpa_headline.parquet')
lpa_age.to_parquet(OUT / 'lpa_age_strat.parquet')
print(f'wrote {OUT}/lpa_*.parquet')

wrote outputs/lpa_*.parquet
